# Generate Netlist Files from NPY Placement Objects

This notebook provides tools to convert .npy placement files into DEF (Design Exchange Format) and JSON netlist files.

## Features:
- Load placement data from .npy files
- Generate DEF files with cell placements
- Generate JSON format netlists
- Batch processing for multiple designs
- Validation and statistics

## 1. Import Required Libraries

In [7]:
import numpy as np
import json
import os
from pathlib import Path
from datetime import datetime
import glob
from tqdm import tqdm

## 2. Configuration and Paths

In [8]:
# Base directories
BASE_DIR = r"H:\Labs\Generative Ai\Ayush1\Ayush"
NPY_DIR_GCELL = os.path.join(BASE_DIR, "CircuitNet", "instance_placement_gcell-001", "instance_placement_gcell")
NPY_DIR_MICRON = os.path.join(BASE_DIR, "CircuitNet", "instance_placement_micron-002", "instance_placement_micron")
OUTPUT_DIR = os.path.join(BASE_DIR, "generated_netlists")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"NPY Directory (GCell): {NPY_DIR_GCELL}")
print(f"NPY Directory (Micron): {NPY_DIR_MICRON}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"\nOutput directory created: {os.path.exists(OUTPUT_DIR)}")

NPY Directory (GCell): H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\instance_placement_gcell-001\instance_placement_gcell
NPY Directory (Micron): H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\instance_placement_micron-002\instance_placement_micron
Output Directory: H:\Labs\Generative Ai\Ayush1\Ayush\generated_netlists

Output directory created: True


## 3. Load and Inspect NPY Files

In [9]:
def load_npy_placement(npy_path):
    """
    Load placement data from a CircuitNet .npy file.

    The files store a Python dict:
        { cell_name (str): [x_min, y_min, x_max, y_max], ... }

    Returns:
        dict  { cell_name: [xmin, ymin, xmax, ymax] }
        or None on error.
    """
    try:
        raw = np.load(npy_path, allow_pickle=True)
        # np.load wraps a dict inside a 0-d object array
        if raw.ndim == 0:
            return raw.item()          # unwrap to plain dict
        # Already a dict (rare)
        if isinstance(raw, dict):
            return raw
        raise ValueError(f"Unexpected data type: {type(raw)}")
    except Exception as e:
        print(f"Error loading {npy_path}: {e}")
        return None

# List available NPY files
gcell_files  = sorted(glob.glob(os.path.join(NPY_DIR_GCELL,  "*.npy")))
micron_files = sorted(glob.glob(os.path.join(NPY_DIR_MICRON, "*.npy")))

print(f"Found {len(gcell_files)} GCell NPY files")
print(f"Found {len(micron_files)} Micron NPY files")
print(f"\nFirst 5 GCell files:")
for f in gcell_files[:5]:
    print(f"  {os.path.basename(f)}")


Found 10242 GCell NPY files
Found 10242 Micron NPY files

First 5 GCell files:
  1-RISCY-a-1-c2-u0.7-m1-p1-f0.npy
  10-RISCY-a-1-c2-u0.7-m2-p2-f0.npy
  100-RISCY-a-1-c2-u0.85-m2-p3-f0.npy
  1000-RISCY-a-2-c5-u0.75-m2-p1-f0.npy
  10000-zero-riscy-b-3-c2-u0.85-m1-p6-f1.npy


## 11. Generate Structural Verilog Netlist (.v)

Builds a **structural Verilog** module from CircuitNet graph data (`node_attr`, `pin_attr`, `net_attr`).  
Output `.v` is accepted by Cadence Innovus / Synopsys ICC2 / OpenROAD as the logical netlist.

In [10]:
"""
=============================================================================
 SECTION 11 — Generate Structural Verilog Netlist (.v)
=============================================================================
Builds a structural Verilog module from CircuitNet graph data:
  node_attr  → cell instance names + cell types
  pin_attr   → pin name, connected net index, owner cell index
  net_attr   → net names

Output .v is accepted by Cadence Innovus / Synopsys ICC2 / OpenROAD
as the logical netlist.
=============================================================================
"""
import re
from collections import defaultdict

# ── Graph data directories ──────────────────────────────────────────────────
GRAPH_DIR     = os.path.join(BASE_DIR, 'CircuitNet', 'graph_features', 'graph_information')
NODE_ATTR_DIR = os.path.join(GRAPH_DIR, 'node_attr')
PIN_ATTR_DIR  = os.path.join(GRAPH_DIR, 'pin_attr')
NET_ATTR_DIR  = os.path.join(GRAPH_DIR, 'net_attr')

# ── Helper: map placement filename → graph design name ─────────────────────
def get_graph_design_name(placement_filename):
    """
    '1-RISCY-a-1-c2-u0.7-m1-p1-f0.npy'  →  'RISCY-a-1-c2'
    Strip leading digits and suffix parameters after the c<N> token.
    """
    name  = os.path.splitext(os.path.basename(str(placement_filename)))[0]
    parts = name.split('-')
    if parts and parts[0].isdigit():
        parts = parts[1:]
    result = []
    for p in parts:
        result.append(p)
        if p.startswith('c') and p[1:].isdigit():
            break
    return '-'.join(result)

# ── Helper: Verilog-safe identifier ────────────────────────────────────────
_SIMPLE_ID = re.compile(r'^[A-Za-z_][A-Za-z0-9_$]*$')
def vid(name):
    """Return a Verilog-safe form of `name`.
    Simple alphanum/underscore names are returned as-is.
    Everything else uses Verilog escaped identifiers: \\name<space>
    """
    s = str(name)
    return s if _SIMPLE_ID.match(s) else f"\\{s} "

# ── Helper: infer pin direction from pin name ───────────────────────────────
_OUT_RE   = re.compile(r'^(Q[NB]?|Z|Y|CO|S[^E]|F|O\d*|DOUT\d*|SO|COUT)$', re.I)
_PWR_RE   = re.compile(r'^(VDD|VCC|VDDCE|VDDPE)$', re.I)
_GND_RE   = re.compile(r'^(VSS|VSSE|GND)$', re.I)
def pin_direction(pin_name):
    n = pin_name.upper()
    if _PWR_RE.match(n): return 'INOUT', 'POWER'
    if _GND_RE.match(n): return 'INOUT', 'GROUND'
    if _OUT_RE.match(n): return 'OUTPUT', 'SIGNAL'
    return 'INPUT', 'SIGNAL'

# ── Main generator ──────────────────────────────────────────────────────────
def generate_verilog_netlist(placement_filename, output_path=None):
    """
    Generate a structural Verilog netlist (.v) for the design.

    Args:
        placement_filename : full path to the .npy placement file  OR  just
                             the basename  OR  the graph design name string.
        output_path        : where to write the .v file.
                             Defaults to OUTPUT_DIR/<design_name>.v
    """
    graph_name   = get_graph_design_name(placement_filename)
    module_name  = re.sub(r'[^A-Za-z0-9_]', '_', graph_name)
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, f"{module_name}.v")

    node_path = os.path.join(NODE_ATTR_DIR, f"{graph_name}_node_attr.npy")
    pin_path  = os.path.join(PIN_ATTR_DIR,  f"{graph_name}_pin_attr.npy")
    net_path  = os.path.join(NET_ATTR_DIR,  f"{graph_name}_net_attr.npy")

    missing = [p for p in (node_path, pin_path, net_path) if not os.path.exists(p)]
    if missing:
        print(f"ERROR — missing graph files:\n  " + "\n  ".join(missing))
        return

    node_attr = np.load(node_path, allow_pickle=True)
    pin_attr  = np.load(pin_path,  allow_pickle=True)
    net_attr  = np.load(net_path,  allow_pickle=True)

    cell_names = list(node_attr[0])        # instance names
    cell_types = list(node_attr[1])        # cell type / master name
    net_names  = list(net_attr[0])         # net names

    # Build: cell_idx → [(pin_name, net_idx), ...]
    cell_pins = defaultdict(list)
    for pi in range(pin_attr.shape[1]):
        p_name   = str(pin_attr[0][pi])
        raw_net  = pin_attr[1][pi]
        raw_cell = pin_attr[2][pi]
        net_idx  = int(raw_net[0]  if isinstance(raw_net,  (list, np.ndarray)) else raw_net)
        cell_idx = int(raw_cell[0] if isinstance(raw_cell, (list, np.ndarray)) else raw_cell)
        cell_pins[cell_idx].append((p_name, net_idx))

    # Top-level ports = nets with no '/' and not inserted by synthesis tools
    port_nets = [
        n for n in net_names
        if '/' not in n
        and not n.startswith(('FE_', 'SYNOPSYS_UNCONNECTED', 'net_'))
    ]

    input_ports, output_ports = [], []
    _clk_rst = re.compile(r'(CLK|CK|CLOCK|RST|RESET|RSTN)', re.I)
    _out_port = re.compile(r'(_O$|_OUT$|_Q$)', re.I)
    for pn in port_nets:
        if _out_port.search(pn):
            output_ports.append(pn)
        else:
            input_ports.append(pn)   # default ports to input

    all_ports = input_ports + output_ports
    port_set  = set(all_ports)
    internal_nets = [n for n in net_names if n not in port_set]

    lines = []
    lines.append(f"// Structural Verilog Netlist")
    lines.append(f"// Design  : {graph_name}")
    lines.append(f"// Cells   : {len(cell_names):,}")
    lines.append(f"// Nets    : {len(net_names):,}")
    lines.append(f"// Ports   : {len(all_ports)}  "
                 f"(in={len(input_ports)}, out={len(output_ports)})")
    lines.append(f"// Generated by Generate_Netlist_from_NPY.ipynb")
    lines.append("")

    # Module declaration
    if all_ports:
        port_list = ",\n    ".join(vid(p) for p in all_ports)
        lines.append(f"module {module_name} (")
        lines.append(f"    {port_list}")
        lines.append(f");")
    else:
        lines.append(f"module {module_name} ();")
    lines.append("")

    # Port directions
    for p in input_ports:
        lines.append(f"  input  {vid(p)};")
    for p in output_ports:
        lines.append(f"  output {vid(p)};")
    if all_ports:
        lines.append("")

    # Internal wire declarations
    lines.append(f"  // Internal wires ({len(internal_nets):,})")
    for n in internal_nets:
        lines.append(f"  wire {vid(n)};")
    lines.append("")

    # Cell instantiations
    lines.append(f"  // Cell instances ({len(cell_names):,})")
    lines.append("")
    for ci, (cname, ctype) in enumerate(zip(cell_names, cell_types)):
        safe_type = re.sub(r'[^A-Za-z0-9_$]', '_', str(ctype))
        pins      = cell_pins.get(ci, [])
        if pins:
            port_map = ", ".join(
                f".{pn}({vid(net_names[ni]) if ni < len(net_names) else ''})"
                for pn, ni in pins
            )
            lines.append(f"  {safe_type} {vid(cname)} ( {port_map} );")
        else:
            lines.append(f"  {safe_type} {vid(cname)} ();")

    lines.append("")
    lines.append(f"endmodule  // {module_name}")

    with open(output_path, 'w') as fh:
        fh.write("\n".join(lines))

    print(f"Verilog netlist written : {output_path}")
    print(f"  Module   : {module_name}")
    print(f"  Cells    : {len(cell_names):,}")
    print(f"  Nets     : {len(net_names):,}")
    print(f"  Ports    : {len(all_ports)}  (inputs={len(input_ports)}, outputs={len(output_ports)})")
    print(f"  Wires    : {len(internal_nets):,}")

print("generate_verilog_netlist() defined.")
print(f"Graph directories configured:")
print(f"  node_attr : {NODE_ATTR_DIR}")
print(f"  pin_attr  : {PIN_ATTR_DIR}")
print(f"  net_attr  : {NET_ATTR_DIR}")


generate_verilog_netlist() defined.
Graph directories configured:
  node_attr : H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\graph_features\graph_information\node_attr
  pin_attr  : H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\graph_features\graph_information\pin_attr
  net_attr  : H:\Labs\Generative Ai\Ayush1\Ayush\CircuitNet\graph_features\graph_information\net_attr


## 11b. Generate Structural Verilog in UART / Cadence Encounter Style

Generates a netlist in the **same format as `netlist.v`** (Cadence Encounter RTL Compiler output):
- Flat comma-separated port list wrapped at ~60 chars
- `input` / `output` port direction declarations after the module header
- Wire declarations grouped **8 per line** (`wire n_0, n_1, n_2, n_3, n_4, n_5, n_6, n_7;`)
- Instance port connections formatted as `.PIN (net)` with 80-char line wrapping
- `UNCONNECTED0`, `UNCONNECTED1`, … wires for unused cell outputs

In [11]:
"""
=============================================================================
 SECTION 11b — Generate Structural Verilog in UART / Cadence Encounter Style
=============================================================================
Output matches the format of netlist.v (Cadence Encounter RTL Compiler):
  - Flat port list wrapped at ~60 chars after the module keyword
  - input / output declarations after the module header
  - Wire declarations grouped 8 per line  (wire n_0, n_1, ..., n_7;)
  - UNCONNECTED0, UNCONNECTED1, ... wires for unused cell outputs
  - Instance connections indented as  .PIN (net)  wrapped at 80 chars
=============================================================================
"""
import textwrap
from collections import defaultdict

# ── Helpers ────────────────────────────────────────────────────────────────
_SIMPLE_ID_RE  = re.compile(r'^[A-Za-z_][A-Za-z0-9_$]*$')
_OUT_PIN_RE    = re.compile(r'^(Q[NB]?|Z|Y|CO|S[^E]|F|O\d*|DOUT\d*|SO|COUT)$', re.I)
_PORT_OUT_RE   = re.compile(r'(_O$|_OUT$|_Q$)', re.I)

def _enc(name):
    """Verilog escaped identifier if needed (backslash prefix, trailing space)."""
    s = str(name)
    return s if _SIMPLE_ID_RE.match(s) else f"\\{s} "

def _wrap_port_list(ports, indent="    ", width=70):
    """Wrap a port list to  width  chars, one entry per comma-break."""
    lines, cur = [], indent
    for i, p in enumerate(ports):
        token = _enc(p) + ("," if i < len(ports) - 1 else "")
        candidate = cur + (" " if cur != indent else "") + token
        if len(candidate) > width and cur != indent:
            lines.append(cur)
            cur = indent + token
        else:
            cur = candidate
    if cur.strip():
        lines.append(cur)
    return "\n".join(lines)

def _chunk_wires(wire_names, per_line=8):
    """Yield groups of up to per_line wire names for grouped declarations."""
    for i in range(0, len(wire_names), per_line):
        yield wire_names[i:i + per_line]


def generate_verilog_uart_style(placement_filename, output_path=None):
    """
    Generate a UART / Cadence Encounter-style structural Verilog netlist.

    Format matches netlist.v:
      module uart(clk, rst_n, ...);
        input clk, rst_n;
        output tx;
        wire n_0, n_1, ..., n_7;
        NAND2XL g123 (.A (net1), .B (net2), .Y (net3));
      endmodule

    Args:
        placement_filename : path to the .npy placement file OR design name
        output_path        : destination .v file
                             (default: OUTPUT_DIR/<module>_uart_style.v)
    """
    graph_name  = get_graph_design_name(placement_filename)
    module_name = re.sub(r'[^A-Za-z0-9_]', '_', graph_name)
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, f"{module_name}_uart_style.v")

    # ── Load graph data ────────────────────────────────────────────────────
    node_path = os.path.join(NODE_ATTR_DIR, f"{graph_name}_node_attr.npy")
    pin_path  = os.path.join(PIN_ATTR_DIR,  f"{graph_name}_pin_attr.npy")
    net_path  = os.path.join(NET_ATTR_DIR,  f"{graph_name}_net_attr.npy")

    missing = [p for p in (node_path, pin_path, net_path) if not os.path.exists(p)]
    if missing:
        print("ERROR — missing graph files:\n  " + "\n  ".join(missing))
        return None

    node_attr = np.load(node_path, allow_pickle=True)
    pin_attr  = np.load(pin_path,  allow_pickle=True)
    net_attr  = np.load(net_path,  allow_pickle=True)

    cell_names = list(node_attr[0])
    cell_types = list(node_attr[1])
    net_names  = list(net_attr[0])

    # ── Build cell → [(pin_name, net_idx)] ────────────────────────────────
    cell_pins = defaultdict(list)
    output_pins_per_cell = defaultdict(list)   # track output pins for UNCONNECTED

    for pi in range(pin_attr.shape[1]):
        pname    = str(pin_attr[0][pi])
        raw_net  = pin_attr[1][pi]
        raw_cell = pin_attr[2][pi]
        net_idx  = int(raw_net[0]  if isinstance(raw_net,  (list, np.ndarray)) else raw_net)
        cidx     = int(raw_cell[0] if isinstance(raw_cell, (list, np.ndarray)) else raw_cell)
        cell_pins[cidx].append((pname, net_idx))
        if _OUT_PIN_RE.match(pname):
            output_pins_per_cell[cidx].append(pname)

    # ── Classify ports (same heuristic as generate_verilog_netlist) ────────
    port_set = set()
    input_ports, output_ports = [], []
    for n in net_names:
        if '/' not in n and not n.startswith(('FE_', 'SYNOPSYS_UNCONNECTED', 'net_')):
            port_set.add(n)
            if _PORT_OUT_RE.search(n):
                output_ports.append(n)
            else:
                input_ports.append(n)

    all_ports      = input_ports + output_ports
    internal_nets  = [n for n in net_names if n not in port_set]

    # ── UNCONNECTED wires (one per disconnected output pin) ────────────────
    # We mark UNCONNECTED nets for output pins whose net name starts with
    # 'SYNOPSYS_UNCONNECTED' or is not connected to anything meaningful.
    unconnected_wires = []
    uc_counter = [0]   # mutable counter

    def get_unconnected():
        name = f"UNCONNECTED{uc_counter[0]}" if uc_counter[0] > 0 else "UNCONNECTED"
        uc_counter[0] += 1
        unconnected_wires.append(name)
        return name

    # Replace SYNOPSYS_UNCONNECTED nets in cell_pins with UNCONNECTED labels
    synopsys_uc_re = re.compile(r'SYNOPSYS_UNCONNECTED', re.I)
    resolved_cell_pins = {}
    for cidx, pins in cell_pins.items():
        new_pins = []
        for pname, nidx in pins:
            if nidx < len(net_names) and synopsys_uc_re.search(str(net_names[nidx])):
                new_pins.append((pname, get_unconnected(), True))   # True = use UNCONNECTED
            else:
                net_label = net_names[nidx] if nidx < len(net_names) else f"net_{nidx}"
                new_pins.append((pname, net_label, False))
        resolved_cell_pins[cidx] = new_pins

    # ── Start writing ─────────────────────────────────────────────────────
    lines = []
    lines.append(f"// Generated by Cadence Encounter(R) RTL Compiler (reproduced)")
    lines.append(f"")
    lines.append(f"// Design : {graph_name}")
    lines.append(f"")

    # Module header — Cadence style flat port list wrapped at 60 chars
    if all_ports:
        port_tokens = [_enc(p) for p in all_ports]
        # Cadence wraps at ~60 chars using 5-space indent after "module name("
        first_line = f"module {module_name}("
        cur = first_line
        port_lines = []
        for i, tok in enumerate(port_tokens):
            sep = ", " if i < len(port_tokens) - 1 else ""
            candidate = cur + tok + sep
            if len(candidate) > 60 and cur != first_line:
                port_lines.append(cur)
                cur = "     " + tok + sep
            else:
                cur = candidate
        port_lines.append(cur + ");")
        lines.extend(port_lines)
    else:
        lines.append(f"module {module_name}();")

    # Port direction declarations  (Cadence groups them on one line each direction)
    if input_ports:
        lines.append(f"  input {', '.join(_enc(p) for p in input_ports)};")
    for op in output_ports:
        lines.append(f"  output {_enc(op)};")

    if all_ports:
        lines.append("")

    # Re-declare port wires (Cadence does this)
    if input_ports:
        lines.append(f"  wire {', '.join(_enc(p) for p in input_ports)};")
    for op in output_ports:
        lines.append(f"  wire {_enc(op)};")
    if all_ports:
        lines.append("")

    # Internal wire declarations  — grouped 8 per line like Cadence
    if internal_nets:
        for chunk in _chunk_wires(internal_nets, per_line=8):
            wire_decl = "  wire " + ", ".join(_enc(n) for n in chunk) + ";"
            lines.append(wire_decl)

    # UNCONNECTED wires (will be appended after instances are processed)
    # We collect them first but write them in a block before instances.
    # Save current position; we'll insert them after the wire block.
    uc_placeholder_idx = len(lines)   # we'll insert here later
    lines.append("")   # spacer

    # ── Cell instances ─────────────────────────────────────────────────────
    for ci, (cname, ctype) in enumerate(zip(cell_names, cell_types)):
        safe_type = re.sub(r'[^A-Za-z0-9_$]', '_', str(ctype))
        pins_for_cell = resolved_cell_pins.get(ci, [])

        if not pins_for_cell:
            lines.append(f"  {safe_type} {_enc(cname)} ();")
            continue

        # Format each port connection as  .PIN (net)
        port_strs = [
            f".{pname} ({_enc(net_lbl)})"
            for pname, net_lbl, _ in pins_for_cell
        ]

        # Build  "  TYPE INST (.A (n1), .B (n2),"  wrapping at 80 chars
        prefix   = f"  {safe_type} {_enc(cname)} ("
        cur_line = prefix + port_strs[0]

        inst_lines = []
        for i, ps in enumerate(port_strs[1:], 1):
            sep   = "," if i < len(port_strs) - 1 else ""
            token = cur_line + ", " + ps
            if len(token) > 80:
                inst_lines.append(cur_line + ",")
                cur_line = "       " + ps
            else:
                cur_line = token
        inst_lines.append(cur_line + ");")

        lines.extend(inst_lines)

    lines.append("")
    lines.append(f"endmodule")

    # Insert UNCONNECTED wire declarations at the saved placeholder position
    if unconnected_wires:
        uc_lines = []
        for chunk in _chunk_wires(unconnected_wires, per_line=8):
            uc_lines.append("  wire " + ", ".join(chunk) + ";")
        # Replace the spacer we inserted
        lines[uc_placeholder_idx] = "\n".join(uc_lines) + "\n"

    with open(output_path, 'w', encoding='utf-8') as fh:
        fh.write("\n".join(lines))

    print(f"UART-style Verilog written : {output_path}")
    print(f"  Module     : {module_name}")
    print(f"  Cells      : {len(cell_names):,}")
    print(f"  Nets       : {len(net_names):,}")
    print(f"  Ports      : {len(all_ports)}  (in={len(input_ports)}, out={len(output_ports)})")
    print(f"  Int. wires : {len(internal_nets):,}")
    print(f"  UNCONN.    : {len(unconnected_wires)}")
    return output_path


# ── Quick demo: generate for first GCell file ──────────────────────────────
if gcell_files:
    test_file  = gcell_files[0]
    graph_name = get_graph_design_name(test_file)
    module_name = re.sub(r'[^A-Za-z0-9_]', '_', graph_name)
    uart_out   = os.path.join(OUTPUT_DIR, f"{module_name}_uart_style.v")
    print(f"Generating UART-style netlist for: {graph_name}\n")
    result = generate_verilog_uart_style(test_file, uart_out)
    if result and os.path.exists(result):
        print(f"\nFirst 30 lines of {os.path.basename(result)}:")
        print("=" * 65)
        with open(result, encoding='utf-8') as fh:
            for i, ln in enumerate(fh):
                if i >= 30:
                    print("  ...")
                    break
                print(ln, end='')
else:
    print("No GCell files found — run the configuration cell first.")


Generating UART-style netlist for: RISCY-a-1-c2

UART-style Verilog written : H:\Labs\Generative Ai\Ayush1\Ayush\generated_netlists\RISCY_a_1_c2_uart_style.v
  Module     : RISCY_a_1_c2
  Cells      : 53,586
  Nets       : 54,734
  Ports      : 1605  (in=1588, out=17)
  Int. wires : 53,129
  UNCONN.    : 405

First 30 lines of RISCY_a_1_c2_uart_style.v:
// Generated by Cadence Encounter(R) RTL Compiler (reproduced)

// Design : RISCY-a-1-c2

module RISCY_a_1_c2(clk, rst_n, rstn_int, testmode_i, 
     \boot_addr_int[8] , \boot_addr_int[9] , 
     \boot_addr_int[10] , \boot_addr_int[11] , 
     \boot_addr_int[12] , \boot_addr_int[13] , 
     \boot_addr_int[14] , \boot_addr_int[15] , 
     \boot_addr_int[16] , \boot_addr_int[17] , 
     \boot_addr_int[18] , \boot_addr_int[19] , 
     \boot_addr_int[20] , \boot_addr_int[21] , 
     \boot_addr_int[22] , \boot_addr_int[23] , 
     \boot_addr_int[24] , \boot_addr_int[25] , 
     \boot_addr_int[26] , \boot_addr_int[27] , 
     \boot_addr_int[2

## 12. Generate LEF Technology / Library File (.lef)

Builds a **LEF** (Library Exchange Format) file with one `MACRO` block per unique cell type.  
Cell sizes are derived from actual placement data; pin directions are inferred from pin names.  
Compatible with **Cadence Innovus**, Synopsys ICC2, and OpenROAD.

In [12]:
"""
=============================================================================
 SECTION 12 — Generate LEF Technology / Library File (.lef)
=============================================================================
Builds a LEF (Library Exchange Format) file Cadence Innovus / OpenROAD can
read as the abstract cell library:

  - MACRO block per unique cell type found in node_attr
  - SIZE    = median width x median height from actual placement instances
  - PINs    = all unique pin names for that cell type, with inferred direction
  - SITE    = CoreSite  (standard cells)  or left blank for big cell types
  - CLASS   = BLOCK (macros / memories / PLLs)  or CORE (standard cells)

Uses the same graph-data directories and helpers defined in Section 11.
=============================================================================
"""
import statistics

# Heuristic: cell types that are "big" / non-standard
_MACRO_TYPE_RE = re.compile(
    r'^(RAM|ROM|MEM|PLL|DLL|PAD|IO|CLOCK|CLK_|FLASH|SRAM|FIFO|BUF_BIG|CLKBUF)',
    re.I
)

def _is_macro_class(cell_type):
    return bool(_MACRO_TYPE_RE.match(str(cell_type)))


def generate_lef_file(placement_filename, output_path=None):
    """
    Generate a LEF abstract library for all unique cell types in the design.

    Args:
        placement_filename : path to the .npy placement file  (or design name).
        output_path        : where to write the .lef file.
                             Defaults to OUTPUT_DIR/<design_name>.lef
    """
    graph_name  = get_graph_design_name(placement_filename)
    module_name = re.sub(r'[^A-Za-z0-9_]', '_', graph_name)
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, f"{module_name}.lef")

    # Load graph files
    node_path = os.path.join(NODE_ATTR_DIR, f"{graph_name}_node_attr.npy")
    pin_path  = os.path.join(PIN_ATTR_DIR,  f"{graph_name}_pin_attr.npy")

    missing = [p for p in (node_path, pin_path) if not os.path.exists(p)]
    if missing:
        print("ERROR - missing graph files:\n  " + "\n  ".join(missing))
        return

    node_attr = np.load(node_path, allow_pickle=True)
    pin_attr  = np.load(pin_path,  allow_pickle=True)

    cell_names = list(node_attr[0])
    cell_types = list(node_attr[1])

    # Load placement for physical sizes.
    # MUST use Micron files (real µm coordinates) NOT GCell files.
    # GCell files store integer grid indices (0-246) — treating them as µm
    # gives wrong sizes (e.g. SIZE 1.0 BY 1.0 for cells that span one grid
    # square, and SIZE 0.0 BY 0.0 for sub-grid-cell standard cells).
    # Micron files give the true physical dimensions (e.g. INV = 0.42 x 1.05 µm).
    micron_files = sorted(glob.glob(os.path.join(NPY_DIR_MICRON, f"*{graph_name}*.npy")))
    gcell_files_match = sorted(glob.glob(os.path.join(NPY_DIR_GCELL, f"*{graph_name}*.npy")))
    # Prefer Micron; only fall back to GCell if no Micron file is available
    npy_files = micron_files if micron_files else gcell_files_match

    placement_dict = {}
    if npy_files:
        placement_dict = load_npy_placement(npy_files[0])
        src = "Micron" if micron_files else "GCell (fallback — sizes may be inaccurate)"
        print(f"  Using {src} placement: {os.path.basename(npy_files[0])}")
    else:
        print("  WARNING: No matching placement NPY file found — using default fallback sizes")

    # type_sizes: {cell_type: [(width, height), ...]}
    # Only record non-zero sizes (zero means the cell was sub-GCell resolution,
    # or was not placed in this particular floorplan variant).
    type_sizes = defaultdict(list)
    for cname, ctype in zip(cell_names, cell_types):
        if cname in placement_dict:
            bbox = placement_dict[cname]
            w = abs(float(bbox[2]) - float(bbox[0]))
            h = abs(float(bbox[3]) - float(bbox[1]))
            if w > 0 and h > 0:
                type_sizes[str(ctype)].append((w, h))

    # type_pins: {cell_type: {pin_name}}
    idx_to_type = {i: str(t) for i, t in enumerate(cell_types)}
    type_pins = defaultdict(set)
    for pi in range(pin_attr.shape[1]):
        p_name   = str(pin_attr[0][pi])
        raw_cell = pin_attr[2][pi]
        cell_idx = int(raw_cell[0] if isinstance(raw_cell, (list, np.ndarray)) else raw_cell)
        if cell_idx < len(cell_types):
            type_pins[idx_to_type[cell_idx]].add(p_name)

    unique_cell_types = sorted(set(str(t) for t in cell_types))

    # Write LEF
    LEF_VERSION   = "5.8"
    LEF_UNITS_DB  = 2000

    lines = []
    lines.append(f"VERSION {LEF_VERSION} ;")
    lines.append('BUSBITCHARS "[]" ;')
    lines.append('DIVIDERCHAR "/" ;')
    lines.append("")
    lines.append("UNITS")
    lines.append(f"  DATABASE MICRONS {LEF_UNITS_DB} ;")
    lines.append("END UNITS")
    lines.append("")

    # Routing layers (minimal definition for Innovus)
    for li, lname in enumerate(["li1", "met1", "met2", "met3", "met4", "met5"], 1):
        direction = "HORIZONTAL" if li % 2 == 0 else "VERTICAL"
        pitch     = round(0.34 * li, 4)
        width     = round(0.17 * li, 4)
        lines.append(f"LAYER {lname}")
        lines.append(f"  TYPE ROUTING ;")
        lines.append(f"  DIRECTION {direction} ;")
        lines.append(f"  PITCH {pitch} ;")
        lines.append(f"  WIDTH {width} ;")
        lines.append(f"END {lname}")
        lines.append("")

    # Site definition
    lines.append("SITE CoreSite")
    lines.append("  CLASS CORE ;")
    lines.append("  SIZE 0.46 BY 2.72 ;")
    lines.append("END CoreSite")
    lines.append("")

    # MACRO blocks - one per unique cell type
    for ctype in unique_cell_types:
        safe_type = re.sub(r'[^A-Za-z0-9_$]', '_', ctype)
        is_macro  = _is_macro_class(ctype)
        cls       = "BLOCK" if is_macro else "CORE"

        sizes = type_sizes.get(ctype, [])
        if sizes:
            widths  = [s[0] for s in sizes]
            heights = [s[1] for s in sizes]
            width   = round(statistics.median(widths),  4)
            height  = round(statistics.median(heights), 4)
        else:
            width, height = 1.0, 2.72   # fallback when no placement data

        pins = sorted(type_pins.get(ctype, set()))

        lines.append(f"MACRO {safe_type}")
        lines.append(f"  CLASS {cls} ;")
        lines.append(f"  SIZE {width} BY {height} ;")
        lines.append(f"  SYMMETRY X Y R90 ;")
        if not is_macro:
            lines.append(f"  SITE CoreSite ;")
        lines.append(f"  ORIGIN 0.0 0.0 ;")
        lines.append("")

        for pname in pins:
            direction, use = pin_direction(pname)
            safe_pin = re.sub(r'[^A-Za-z0-9_$\[\]]', '_', pname)
            pin_rect = min(width, 0.14)
            lines.append(f"  PIN {safe_pin}")
            lines.append(f"    DIRECTION {direction} ;")
            lines.append(f"    USE {use} ;")
            lines.append(f"    PORT")
            lines.append(f"      LAYER met1 ;")
            lines.append(f"        RECT 0.0 0.0 {pin_rect:.4f} {pin_rect:.4f} ;")
            lines.append(f"    END")
            lines.append(f"  END {safe_pin}")
            lines.append("")

        lines.append(f"END {safe_type}")
        lines.append("")

    lines.append("END LIBRARY")

    with open(output_path, 'w') as fh:
        fh.write("\n".join(lines))

    print(f"LEF file written       : {output_path}")
    print(f"  Unique cell types    : {len(unique_cell_types):,}")
    print(f"  Types with sizes     : {len(type_sizes):,}")
    print(f"  Total pins defined   : {sum(len(v) for v in type_pins.values()):,}")

print("generate_lef_file() defined.")


generate_lef_file() defined.


## 13. Run — Generate Verilog + LEF for a Design

Calls `generate_verilog_netlist()` and `generate_lef_file()` on the first GCell NPY file and previews the output.  
**Run cells 11 and 12 first** to define the generator functions.

In [13]:
# ============================================================
# 13. RUN — Generate Verilog + LEF for the first GCell design
# ============================================================
if gcell_files:
    test_file   = gcell_files[0]
    graph_name  = get_graph_design_name(test_file)
    module_name = re.sub(r'[^A-Za-z0-9_]', '_', graph_name)

    verilog_out = os.path.join(OUTPUT_DIR, f"{module_name}.v")
    lef_out     = os.path.join(OUTPUT_DIR, f"{module_name}.lef")

    print(f"Design  : {graph_name}")
    print(f"NPY     : {os.path.basename(test_file)}")
    print(f"Outputs : {OUTPUT_DIR}")
    print()

    print("=" * 60)
    print("Generating Verilog netlist...")
    print("=" * 60)
    generate_verilog_netlist(test_file, verilog_out)

    print()
    print("=" * 60)
    print("Generating LEF library...")
    print("=" * 60)
    generate_lef_file(test_file, lef_out)

    print()
    print("=" * 60)
    print("DONE — file previews")
    print("=" * 60)

    # Preview first 25 lines of Verilog
    if os.path.exists(verilog_out):
        print(f"\n--- {os.path.basename(verilog_out)} (first 25 lines) ---")
        with open(verilog_out) as fh:
            for i, ln in enumerate(fh):
                if i >= 25:
                    print("  ...")
                    break
                print(f"  {ln}", end='')

    # Preview first 40 lines of LEF
    if os.path.exists(lef_out):
        print(f"\n--- {os.path.basename(lef_out)} (first 40 lines) ---")
        with open(lef_out) as fh:
            for i, ln in enumerate(fh):
                if i >= 40:
                    print("  ...")
                    break
                print(f"  {ln}", end='')
else:
    print("No GCell files found. Run cell 2 (configuration) first.")


Design  : RISCY-a-1-c2
NPY     : 1-RISCY-a-1-c2-u0.7-m1-p1-f0.npy
Outputs : H:\Labs\Generative Ai\Ayush1\Ayush\generated_netlists

Generating Verilog netlist...
Verilog netlist written : H:\Labs\Generative Ai\Ayush1\Ayush\generated_netlists\RISCY_a_1_c2.v
  Module   : RISCY_a_1_c2
  Cells    : 53,586
  Nets     : 54,734
  Ports    : 1605  (inputs=1588, outputs=17)
  Wires    : 53,129

Generating LEF library...
  Using Micron placement: 1-RISCY-a-1-c2-u0.7-m1-p1-f0.npy
LEF file written       : H:\Labs\Generative Ai\Ayush1\Ayush\generated_netlists\RISCY_a_1_c2.lef
  Unique cell types    : 153
  Types with sizes     : 142
  Total pins defined   : 673

DONE — file previews

--- RISCY_a_1_c2.v (first 25 lines) ---
  // Structural Verilog Netlist
  // Design  : RISCY-a-1-c2
  // Cells   : 53,586
  // Nets    : 54,734
  // Ports   : 1605  (in=1588, out=17)
  // Generated by Generate_Netlist_from_NPY.ipynb
  
  module RISCY_a_1_c2 (
      clk,
      rst_n,
      rstn_int,
      testmode_i,
   